In [ ]:
pip install crewai

In [ ]:
import os
from getpass import getpass
from crewai import LLM,Agent,Task

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ").strip()
llm = LLM(model="gpt-4o",temperature=0,)
print("OpenAI model configured: gpt-4o")

# **Create Agent 1 — HR Policy Agent**

In [ ]:
hr_policy_agent = Agent(
    role="HR Policy Specialist",
    goal=(
        "Analyze employee HR policies and determine "
        "the rules applicable to the employee's situation."
    ),
    backstory=(
        "You are an HR policy specialist who understands "
        "company policies related to notice periods, leave, "
        "resignation and employee benefits."
    ),
    llm=llm,
    verbose=True
)

In [ ]:
hr_policy_task = Task(
    description=(
        "Analyze the employee's request: {employee_request}. "
        "Identify the applicable notice period, leave policy "
        "and resignation rules relevant to the request."
    ),
    expected_output=(
        "A clear summary of the applicable HR policies, "
        "including notice period, leave rules and "
        "resignation requirements."
    ),
    agent=hr_policy_agent
)

# **Create Agent 2 — Payroll Agent**

In [ ]:
payroll_agent = Agent(
    role="Payroll Specialist",
    goal=(
        "Calculate the employee's final financial settlement "
        "based on salary, leave balance, deductions and "
        "other applicable payments."
    ),
    backstory=(
        "You are a payroll specialist experienced in "
        "employee settlements, salary calculations, "
        "leave encashment and deductions."
    ),
    llm=llm,
    verbose=True
)

In [ ]:
payroll_task = Task(
    description=(
        "Analyze the employee's request: {employee_request}. "
        "Calculate the employee's final settlement using "
        "salary, leave balance, applicable encashment, "
        "deductions and pending reimbursements, if requested."
    ),
    expected_output=(
        "A clear breakdown of salary, leave encashment, "
        "deductions, reimbursements and final settlement, "
        "where relevant and supported by the available information."
    ),
    agent=payroll_agent
)

# **Create Agent 3 — Compliance Agent**

In [ ]:
compliance_agent = Agent(
    role="HR Compliance Specialist",
    goal=(
        "Identify all employee exit procedures, clearances "
        "and required documents before the employee's "
        "last working day."
    ),
    backstory=(
        "You are an HR compliance specialist responsible "
        "for employee exit procedures, clearances and "
        "required documentation."
    ),
    llm=llm,
    verbose=True
)

In [ ]:
compliance_task = Task(
    description=(
        "Analyze the employee's request: {employee_request}. "
        "Identify relevant exit procedures, clearances "
        "and required documents before the employee's "
        "last working day, if requested."
    ),
    expected_output=(
        "A complete employee exit checklist containing "
        "relevant procedures, clearances and documents."
    ),
    agent=compliance_agent
)

# **Create Agent 4 — Documentation Agent**

In [ ]:
documentation_agent = Agent(
    role="HR Documentation Specialist",
    goal=(
        "Prepare professional resignation and employee "
        "handover documents using the information provided "
        "by the other HR specialists."
    ),
    backstory=(
        "You are an HR documentation specialist who creates "
        "clear and professional resignation letters, "
        "handover checklists and HR communications."
    ),
    llm=llm,
    verbose=True
)

In [ ]:
documentation_task = Task(
    description=(
        "Analyze the employee's request: {employee_request}. "
        "Prepare the relevant resignation letter, "
        "handover checklist or HR communication requested "
        "by the employee."
    ),
    expected_output=(
        "A professional resignation letter, handover "
        "checklist or HR communication, as requested."
    ),
    agent=documentation_agent
)

# **Main Orchestrator AGENT — The HR Manager**

In [ ]:
hr_manager = Agent(
    role="HR Manager / Orchestrator",
    goal=(
        "Coordinate specialized HR agents to resolve the "
        "employee's request and produce a complete final response."
    ),
    backstory=(
        "You are an experienced HR manager responsible for "
        "coordinating HR policy, payroll, compliance and "
        "documentation specialists."
    ),
    llm=llm,
    verbose=True
)

In [ ]:
final_hr_task = Task(
    description=(
        "Resolve the employee's request: {employee_request}. "
        "Coordinate the relevant HR policy, payroll, "
        "compliance and documentation specialists as needed. "
        "Provide a final response focused on what the employee requested."
    ),
    expected_output=(
        "A clear, relevant response to the employee's request, "
        "including only the applicable HR information."
    ),
    agent=hr_manager
)

# **Create the Agent Crew**

In [ ]:
from crewai import Crew, Process

def create_hr_crew(user_input):

    request = user_input.lower()

    if any(word in request for word in [
        "notice period", "leave policy", "hr policy", "benefits"
    ]):
        selected_agent = hr_policy_agent
        selected_task = hr_policy_task

    elif any(word in request for word in [
        "salary", "payroll", "final settlement", "encashment", "deduction"
    ]):
        selected_agent = payroll_agent
        selected_task = payroll_task

    elif any(word in request for word in [
        "clearance", "exit procedure", "company property", "last working day"
    ]):
        selected_agent = compliance_agent
        selected_task = compliance_task

    elif any(word in request for word in [
        "resignation letter", "write a letter", "handover document", "draft an email"
    ]):
        selected_agent = documentation_agent
        selected_task = documentation_task

    else:
        selected_agent = hr_policy_agent
        selected_task = hr_policy_task

    hr_crew = Crew(
        agents=[selected_agent],
        tasks=[selected_task],
        process=Process.sequential,
        verbose=True
    )

    return hr_crew

In [ ]:
user_input = input("Enter your HR question: ")

hr_crew = create_hr_crew(user_input)

result = await hr_crew.kickoff_async(
    inputs={"employee_request": user_input}
)

print("\nFinal HR Response:")
print(result)

Enter your HR question: What is the Resignation policy ?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ff59a53e-a554-46cd-87ac-edd3231de93c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the employee's request: What is the Resignation policy ?. Identify the applicable notice         │
│  period, leave policy and resignation rules relevant to the request.                                            │
│  ID: 53a5b593-adaf-4906-af1d-43f0469dc6d8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HR Policy Specialist                                                                                    │
│                                                                                                                 │
│  Task: Analyze the employee's request: What is the Resignation policy ?. Identify the applicable notice         │
│  period, leave policy and resignation rules relevant to the request.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HR Policy Specialist                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To analyze the employee's request regarding the resignation policy, we need to consider the following key      │
│  aspects: notice period, leave policy, and resignation requirements. Here is a comprehensive overview of these  │
│  elements:                                                                                                      │
│                                                                                                                 │
│  1. **Notice Period:**                                                                                          │
│     - The notice period is the duration an employee is required to serve after submitting their resignation     │
│  before they officially leave the company. This period allows the company to manage the transition and find a   │
│  suitable replacement.                                                                                          │
│     - Typically, the notice period is specified in the employment contract or the company's HR policy. It can   │
│  vary based on the employee's role, length of service, and level within the organization.                       │
│     - Common notice periods range from two weeks to three months. For example, entry-level positions might      │
│  require a two-week notice, while managerial roles might require a one to three-month notice.                   │
│                                                                                                                 │
│  2. **Leave Policy:**                                                                                           │
│     - During the notice period, employees may have accrued leave, such as annual leave or vacation days. The    │
│  company's leave policy will dictate how this leave can be utilized.                                            │
│     - Some companies allow employees to take their accrued leave during the notice period, while others may     │
│  require the employee to work through the notice period and compensate them for any unused leave.               │
│     - It's important to check if the company has any specific rules about taking leave during the notice        │
│  period, as some organizations may restrict leave usage to ensure a smooth transition.                          │
│                                                                                                                 │
│  3. **Resignation Requirements:**                                                                               │
│     - Employees are generally required to submit a formal resignation letter, indicating their intention to     │
│  leave the company and specifying their last working day, based on the notice period.                           │
│     - The resignation letter should be addressed to the employee's immediate supervisor or the HR department,   │
│  as per the company's policy.                                                                                   │
│     - Some companies may have additional requirements, such as exit interviews or clearance processes, which    │
│  need to be completed before the employee's departure.                                                          │
│                                                                                                                 │
│  In summary, the applicable HR policies for an employee

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze the employee's request: What is the Resignation policy ?. Identify the applicable notice         │
│  period, leave policy and resignation rules relevant to the request.                                            │
│  Agent: HR Policy Specialist                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final HR Response:
To analyze the employee's request regarding the resignation policy, we need to consider the following key aspects: notice period, leave policy, and resignation requirements. Here is a comprehensive overview of these elements:

1. **Notice Period:**
   - The notice period is the duration an employee is required to serve after submitting their resignation before they officially leave the company. This period allows the company to manage the transition and find a suitable replacement.
   - Typically, the notice period is specified in the employment contract or the company's HR policy. It can vary based on the employee's role, length of service, and level within the organization.
   - Common notice periods range from two weeks to three months. For example, entry-level positions might require a two-week notice, while managerial roles might require a one to three-month notice.

2. **Leave Policy:**
   - During the notice period, employees may have accrued leave, such as a

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ff59a53e-a554-46cd-87ac-edd3231de93c                                                                       │
│  Final Output: To analyze the employee's request regarding the resignation policy, we need to consider the      │
│  following key aspects: notice period, leave policy, and resignation requirements. Here is a comprehensive      │
│  overview of these elements:                                                                                    │
│                                                                                                                 │
│  1. **Notice Period:**                                                                                          │
│     - The notice period is the duration an employee is required to serve after submitting their resignation     │
│  before they officially leave the company. This period allows the company to manage the transition and find a   │
│  suitable replacement.                                                                                          │
│     - Typically, the notice period is specified in the employment contract or the company's HR policy. It can   │
│  vary based on the employee's role, length of service, and level within the organization.                       │
│     - Common notice periods range from two weeks to three months. For example, entry-level positions might      │
│  require a two-week notice, while managerial roles might require a one to three-month notice.                   │
│                                                                                                                 │
│  2. **Leave Policy:**                                                                                           │
│     - During the notice period, employees may have accrued leave, such as annual leave or vacation days. The    │
│  company's leave policy will dictate how this leave can be utilized.                                            │
│     - Some companies allow employees to take their accrued leave during the notice period, while others may     │
│  require the employee to work through the notice period and compensate them for any unused leave.               │
│     - It's important to check if the company has any specific rules about taking leave during the notice        │
│  period, as some organizations may restrict leave usage to ensure a smooth transition.                          │
│                                                                                                                 │
│  3. **Resignation Requirements:**                                                                               │
│     - Employees are generally required to submit a formal resignation letter, indicating their intention to     │
│  leave the company and specifying their last working day, based on the notice period.                           │
│     - The resignation letter should be addressed to the employee's immediate supervisor or the HR department,   │
│  as per the company's policy.                                                                                   │
│     - Some companies may have additional requirements, such as exit interviews or clearance processes, which    │
│  need to be completed before the employee's departure.                                                          │
│                                                                                                                 │
│  In summary, the applicable HR policies for an employe

# **THE END.**